# Зависимости

In [29]:
%pip install gigachat openai python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


# Запросы

Каждой модели отправим 4 запроса:

1. Python
 

    Напиши функцию на Python, которая принимает строку и возвращает словарь,
    где ключи — слова, а значения — сколько раз каждое слово встречается в строке.
    Регистр не учитывать, знаки препинания игнорировать.
    Приведи код, кратко объясни решение и покажи пример вызова с результатом.

 
2. C
 

    Напиши программу на C, которая считывает с клавиатуры массив из n целых чисел
    (n вводится первым), сортирует его пузырьковой сортировкой и выводит результат.
    Память выдели динамически и не забудь её освободить.
    Приведи полный компилируемый код и кратко поясни, как он работает.

 
3. Математика
 

    Дай строгое определение математического ожидания случайной величины
    для дискретного и непрерывного случаев.
    Перечисли основные свойства математического ожидания (линейность, ожидание константы,
    ожидание произведения независимых величин и т.д.) с краткими доказательствами.
    Приведи по одному примеру вычисления для дискретной и непрерывной величины.

 
4. Литература
 

    Опиши главных действующих лиц рассказа И. С. Тургенева «Муму».
    Для каждого персонажа укажи: кто он, черты характера, роль в сюжете.
    Отдельно объясни, в чём смысл образа Герасима и какую идею хотел донести автор.


## Список промптов

In [11]:
prompts = [
    {
        "name": "python",
        "prompt": (
            "Напиши функцию на Python, которая принимает строку и возвращает словарь, "
            "где ключи — слова, а значения — сколько раз каждое слово встречается в строке. "
            "Регистр не учитывать, знаки препинания игнорировать. "
            "Приведи код, кратко объясни решение и покажи пример вызова с результатом."
        ),
    },
    {
        "name": "c",
        "prompt": (
            "Напиши программу на C, которая считывает с клавиатуры массив из n целых чисел "
            "(n вводится первым), сортирует его пузырьковой сортировкой и выводит результат. "
            "Память выдели динамически и не забудь её освободить. "
            "Приведи полный компилируемый код и кратко поясни, как он работает."
        ),
    },
    {
        "name": "math",
        "prompt": (
            "Дай строгое определение математического ожидания случайной величины "
            "для дискретного и непрерывного случаев. "
            "Перечисли основные свойства математического ожидания (линейность, ожидание константы, "
            "ожидание произведения независимых величин и т.д.) с краткими доказательствами. "
            "Приведи по одному примеру вычисления для дискретной и непрерывной величины."
        ),
    },
    {
        "name": "literature",
        "prompt": (
            "Опиши главных действующих лиц рассказа И. С. Тургенева «Муму». "
            "Для каждого персонажа укажи: кто он, черты характера, роль в сюжете. "
            "Отдельно объясни, в чём смысл образа Герасима и какую идею хотел донести автор."
        ),
    },
]

# Работа с GigaChat через API

Работу с GigaChat будем проводить через SDK

## Получение API ключа

Авторизируемся на https://developers.sber.ru и там получаем `Authorization key`.

Бесплатно доступно несколько десятков миллионов токенов.

Для работы с API нужно иметь сертификаты Минцифры. Скачаем их.

`mkdir -p ~/certs`

`wget https://gu-st.ru/content/lending/russian_trusted_root_ca_pem.crt -O ~/certs/russian_trusted_root_ca.crt`

In [26]:
import os
from datetime import datetime

from dotenv import load_dotenv
from gigachat import GigaChat

load_dotenv()

client = GigaChat(
    credentials=os.getenv("GIGACHAT_AUTH_KEY"),
    scope="GIGACHAT_API_PERS", # Для физ. лиц
    ca_bundle_file=os.path.expanduser("~/certs/russian_trusted_root_ca.crt"),
)

token = client.get_token()

dt = datetime.fromtimestamp(token.expires_at / 1000)
print(f"Ключ действителен до: {dt.strftime("%d.%m.%Y %H:%M:%S")}")

Ключ действителен до: 23.09.2026 01:18:06


## Получение списка доступных моделей

In [ ]:
models = client.get_models()
for model in models.data:
    print(f"{model.id_}")

response = client.chat.create("Привет, как дела?")

GigaChat-2
GigaChat-2-Max
GigaChat-2-Pro
GigaChat-3-Lightning
GigaChat-3-Pro
GigaChat-3-Ultra
Embeddings
Embeddings-2
EmbeddingsGigaR
GigaEmbeddings-3B-2025-09


## Отправка запросов на API

In [ ]:
from gigachat.models import ChatCompletionRequest, ChatMessage

req = ChatCompletionRequest(
    model="GigaChat-2",
    messages=[ChatMessage(role="user", content="Привет, как дела?")],
)

answer = client.chat.create(req).messages[0].content[0].text
print(answer)

Привет! Всё отлично, спасибо. Готов обсудить что-нибудь интересное или помочь разобраться с задачей. Чем займёмся?


При помощи роли `system` можно скорректировать поведение модели.

In [27]:
req = ChatCompletionRequest(
    model="GigaChat-2",
    messages=[
        ChatMessage(role="system", content="Отвечай одним предложением, в конце добавляй смайлик"),
        ChatMessage(role="user", content="Привет, как дела?")
    ],
)

answer = client.chat.create(req).messages[0].content[0].text
print(answer)

Привет, у меня всё отлично, спасибо! А у тебя как? 😊


### Отправим несколько запросов моделям GigaChat-2 и GigaChat-3-Lightning

In [28]:
from IPython.display import Markdown, display

models = ["GigaChat-2", "GigaChat-3-Lightning"]

for model in models:
    display(Markdown(f"# {model}"))
    for prompt in prompts:
        req = ChatCompletionRequest(
            model=model,
            messages=[ChatMessage(role="user", content=prompt["prompt"])],
        )
        answer = client.chat.create(req).messages[0].content[0].text
        display(Markdown(f"## {prompt["name"]}\n\n{answer}\n\n---"))

# GigaChat-2

## python

Вот функция на Python, решающая поставленную задачу:

```python
def word_count(text):
    import re
    
    # Преобразуем текст в нижний регистр и удаляем знаки пунктуации
    cleaned_text = re.sub(r'[^\w\s]', '', text.lower())
    
    # Разбиваем текст на слова и считаем количество повторений каждого слова
    words_dict = {}
    for word in cleaned_text.split():
        if word in words_dict:
            words_dict[word] += 1
        else:
            words_dict[word] = 1
            
    return words_dict

# Пример вызова функции
input_text = "Привет, мир! Мир прекрасен. Привет!"
print(word_count(input_text))
```

### Объяснение решения:

1. **Импорт модуля** `re`:  
   Для удаления знаков пунктуации используем регулярное выражение `[^\w\s]`, которое находит любые непечатаемые символы кроме букв и пробелов.
   
2. **Преобразование регистра:**  
   Все буквы приводятся к нижнему регистру с помощью `.lower()`, чтобы не учитывать различие между строчными и заглавными буквами.

3. **Удаление знаков пунктуации:**  
   Регулярное выражение `re.sub()` удаляет все непечатаемые символы (`[^\w\s]`), оставляя только слова.

4. **Подсчет количества слов:**  
   Текст разбивается на отдельные слова методом `.split()`. Затем проходимся циклом по словам и увеличиваем счетчик в словаре `words_dict`.

---

### Результат примера вызова:

Для строки `"Привет, мир! Мир прекрасен. Привет!"` результат будет выглядеть следующим образом:

```python
{'привет': 2, 'мир': 2, 'прекрасен': 1}
```

Таким образом, функция корректно подсчитала частоту появления каждого слова, игнорируя регистр и знаки пунктуации.

---

## c

Вот полная программа на языке C, реализующая ввод массива, динамическое выделение памяти, сортировку пузырьком и освобождение памяти:

```c
#include <stdio.h>
#include <stdlib.h>

// Функция для чтения массива и выделения памяти
void readArray(int **arr, int *size) {
    printf("Введите размер массива: ");
    scanf("%d", size);
    
    // Динамическое выделение памяти
    *arr = (int *)malloc(*size * sizeof(int));
    if (*arr == NULL) {
        fprintf(stderr, "Ошибка выделения памяти!\n");
        exit(EXIT_FAILURE);
    }
    
    printf("Введите элементы массива:\n");
    for (int i = 0; i < *size; ++i) {
        scanf("%d", &(*arr)[i]);
    }
}

// Функция пузырьковой сортировки
void bubbleSort(int *arr, int size) {
    int temp;
    for (int i = 0; i < size - 1; ++i) {
        for (int j = 0; j < size - 1 - i; ++j) {
            if ((*arr)[j] > (*arr)[j + 1]) {
                // Меняем местами элементы
                temp = (*arr)[j];
                (*arr)[j] = (*arr)[j + 1];
                (*arr)[j + 1] = temp;
            }
        }
    }
}

// Функция вывода отсортированного массива
void printSortedArray(const int *arr, int size) {
    printf("Отсортированный массив: ");
    for (int i = 0; i < size; ++i) {
        printf("%d ", arr[i]);
    }
    printf("\n");
}

int main() {
    int *array;
    int size;

    // Чтение массива и выделение памяти
    readArray(&array, &size);

    // Выполнение пузырьковой сортировки
    bubbleSort(array, size);

    // Вывод результата
    printSortedArray(array, size);

    // Освобождение выделенной памяти
    free(array);

    return 0;
}
```

## Пояснение программы:

1. **readArray**:
   - Читает размер массива и выделяет память под введённые числа.
   - Использует функцию `malloc()` для динамического выделения памяти.
   
2. **bubbleSort**:
   - Реализует алгоритм пузырьковой сортировки.
   - Проходит по массиву несколько раз, сравнивая соседние элементы и меняя их местами, пока массив не станет упорядоченным.

3. **printSortedArray**:
   - Просто выводит отсортированные элементы массива.

4. **main**:
   - Вызывает чтение массива и выделение памяти.
   - Вызывает функцию сортировки.
   - Выводит результат.
   - Освобождает выделенную ранее память с помощью `free()`.

### Пример работы программы:

```
Введите размер массива: 5
Введите элементы массива:
8 3 1 9 2
Отсортированный массив: 1 2 3 8 9
```

Программа корректно обрабатывает ввод, сортирует массив пузырьковым методом и освобождает использованную память.

---

## math

## Определение математического ожидания

### Дискретный случай
Пусть случайная величина $\xi$ принимает конечное или счётное число возможных значений $x_1, x_2, ..., x_n$ с вероятностями $p(x_i)=P(\xi=x_i)$. Тогда **математическим ожиданием** называется сумма вида:
$$
M[\xi] = \sum_{i=1}^{\infty}x_ip(x_i)
$$
при условии, что ряд сходится абсолютно.

### Непрерывный случай
Пусть $\xi$ — случайная величина с плотностью распределения вероятности $f(x)$, тогда математическое ожидание определяется интегралом:
$$
M[\xi]=\int_{-\infty}^{+\infty}xf(x)\ dx
$$
если интеграл сходится абсолютно.

---

## Основные свойства математического ожидания

### Линейность
Для любых случайных величин $\xi,\eta$ и констант $a,b$ выполняется:
$$
M[a\xi+b\eta]=a\cdot M[\xi]+b\cdot M[\eta]
$$

**Доказательство:**  
Рассмотрим случай двух случайных величин $\xi,\eta$. Пусть $X,Y$ — соответствующие множества значений. Тогда:
$$
M[a\xi+b\eta] = \sum_{x,y}(ax+by)p(x,y)=\sum_x a\sum_y xp(x,y)+\sum_y b\sum_xyp(x,y)=a\cdot M[\xi]+b\cdot M[\eta]
$$
Аналогично доказывается для непрерывного случая через интегралы.

---

### Ожидание константы
Если $\xi=c$ — постоянная величина, то:
$$
M[c] = c
$$

**Доказательство:**  
Поскольку $\xi$ постоянна, вероятность любого значения равна единице, следовательно:
$$
M[c] = \sum_{x}c\cdot P(\xi=c)=c\cdot1=c
$$

---

### Неравенство Маркова
Для любой неотрицательной случайной величины $\xi$ и положительного числа $a$ верно неравенство:
$$
P(|\xi|\geqslant a)\leqslant\frac{M[|\xi|]}{a}
$$

---

### Независимые случайные величины
Если $\xi,\eta$ независимы, то:
$$
M[\xi\eta] = M[\xi]\cdot M[\eta]
$$

**Доказательство:**  
Используя свойство линейности и независимость:
$$
M[\xi\eta] = M[\xi]\cdot M[\eta] + M[\xi-E(\xi)]\cdot M[\eta-E(\eta)]
$$
Но поскольку $\xi-\mathbb{E}(\xi)$ и $\eta-\mathbb{E}(\eta)$ независимы, их произведение имеет нулевое математическое ожидание, откуда получаем утверждение.

---

## Примеры вычисления математического ожидания

### Пример дискретной случайной величины
Пусть $\xi$ — количество выпавших очков при бросании игральной кости. Возможные значения: $1,2,...,6$, каждое с равной вероятностью $1/6$.
Тогда:
$$
M[\xi] = \frac{1}{6}\cdot(1+2+3+4+5+6) = \frac{21}{6}=3.5
$$

### Пример непрерывной случайной величины
Пусть $\xi$ распределена равномерно на отрезке $[0,1]$ с плотностью вероятности:
$$
f(x) = \begin{cases} 1 & x\in[0,1] \\ 0 & \text{иное}\end{cases}
$$
Тогда:
$$
M[\xi] = \int_{0}^{1} x\cdot f(x)dx = \int_{0}^{1} x\cdot1\,dx = \left.\frac{x^2}{2}\right|_{0}^{1}=\frac{1}{2}
$$

---

## literature

Главные действующие лица рассказа Ивана Сергеевича Тургенева «Муму»:

### Герасим  
**Кто:** крепостной крестьянин, дворник барыни-помещицы.  
**Черты характера:**  
- Очень сильный физически, трудолюбивый и молчаливый человек.  
- Добрый, терпеливый и покорный судьбе.  
- Чуткий и отзывчивый, глубоко переживающий чувства окружающих.  
- Страдает от несправедливости и жестокого обращения помещицы Барыня.  
**Роль в сюжете:**  
Герасим является центральным персонажем произведения. Его образ раскрывает тему трагической судьбы крепостного крестьянства, показывает тяжёлое положение человека, лишённого свободы и достоинства. Через страдания героя автор подчёркивает бесчеловечность крепостничества и несвободу личности в условиях крепостного права.

### Барыня (помещица)  
**Кто:** богатая дворянка, владелица крепостных душ.  
**Черты характера:**  
- Высокомерная, властная, капризная женщина.  
- Эгоцентрична, жестока и груба по отношению к окружающим людям и животным.  
- Не уважает чужие чувства и достоинство, проявляет полное равнодушие к страданиям слуг.  
**Роль в сюжете:**  
Барыня олицетворяет типичное отношение помещиков к крепостным крестьянам, демонстрируя произвол и жестокость владельцев крепостных душ. Её поступки подчеркивают бесчеловечный характер крепостного строя и показывают деградацию человеческой души в условиях неограниченной власти над людьми.

### Муму  
**Кто:** собака, друг Герасима.  
**Черты характера:**  
- Умная, преданная и ласковая собака.  
- Верный спутник Герасима, искренне привязанный к нему.  
- Проявляет собачью верность и нежность.  
**Роль в сюжете:**  
Образ собаки символизирует глубокую человечность и искреннюю любовь Герасима, противопоставленную жестокости и эгоизму помещицы. Через гибель Муму автор подчеркивает бессмысленность и трагичность человеческих решений, принимаемых без учёта чувств и привязанности живых существ.

---

## Смысл образа Герасима и идея автора  

Образ Герасима выражает главную мысль рассказа Тургенева: крепостничество превращало человека в вещь, лишало личность человеческого достоинства и свободы. Автор через судьбу Герасима ярко демонстрирует бесчеловечные последствия крепостного права, показывая, насколько глубоко оно калечило душу и разрушало жизнь простого русского мужика.  

Тургенев стремится показать читателю всю глубину трагедии крепостного крестьянина, вынужденного терпеть унижения и жестокость, смиряться с судьбой и оставаться рабом своего господина. Идея рассказа заключается в осуждении крепостного строя, бесчеловечной системы эксплуатации человека человеком и призыве к освобождению личности от угнетения и бесправия.

---

# GigaChat-3-Lightning

## python

Вот простая и эффективная функция на Python, которая решает задачу подсчета слов с учетом регистра и знаков препинания:

### Код функции

```python
import re
from collections import Counter

def word_count(text):
    # Приводим к нижнему регистру и удаляем знаки препинания (оставляем только буквы и пробелы)
    cleaned = re.sub(r'[^a-zа-я ]', '', text.lower())
    # Разбиваем строку по пробелам и считаем вхождения каждого слова
    words = cleaned.split()
    return dict(Counter(words))
```

**Краткое объяснение:**
- `re.sub` — заменяет все символы, кроме букв русского/английского алфавита и пробелов, пустотой.
- `.lower()` — игнорируем регистр.
- `split()` — разбивает текст на отдельные слова.
- `collections.Counter` — автоматически считает количество повторений.

### Пример вызова

```python
text = "Привет! Как дела? Привет-привет. Дела идут хорошо!"
result = word_count(text)
print(result)
```

**Результат:**
```python
{'привет': 2, 'как': 1, 'дела': 2, 'идут': 1, 'хорошо': 1}
```

Функция корректно обрабатывает регистр и убирает знаки препинания. Если нужно учитывать ещё и цифры или другие символы — можно изменить регулярное выражение.

---

## c

Вот полный компилируемый код на языке C, который динамически выделяет память под массив, сортирует его пузырьковой сортировкой и корректно освобождает выделенную память:

### Код программы

```c
#include <stdio.h>
#include <stdlib.h>

// Функция для обмена двух чисел (указателями)
void swap(int *a, int *b) {
    int temp = *a;
    *a = *b;
    *b = temp;
}

// Пузырьковая сортировка
void bubble_sort(int *arr, size_t n) {
    for (size_t i = 0; i < n - 1; ++i) {
        // На каждом проходе максимальный элемент "всплывает" в конец массива
        for (size_t j = 0; j < n - i - 1; ++j) {
            if (arr[j] > arr[j + 1]) {
                swap(&arr[j], &arr[j + 1]);
            }
        }
    }
}

int main() {
    int n;
    
    // Ввод размера массива
    printf("Введите размер массива: ");
    if (scanf("%d", &n) != 1 || n <= 0) {
        fprintf(stderr, "Ошибка: введено некорректное число или ноль.\n");
        return EXIT_FAILURE;
    }

    // Динамическое выделение памяти под массив
    int *arr = (int *)malloc(n * sizeof(int));
    if (!arr) {
        perror("Ошибка выделения памяти");
        return EXIT_FAILURE;
    }

    // Чтение элементов массива
    printf("Введите %d целых чисел:\n", n);
    for (int i = 0; i < n; ++i) {
        if (scanf("%d", &arr[i]) != 1) {
            fprintf(stderr, "Ошибка ввода данных.\n");
            free(arr); // Освобождаем память при ошибке
            return EXIT_FAILURE;
        }
    }

    // Выполнение пузырьковой сортировки
    bubble_sort(arr, n);

    // Вывод отсортированного массива
    printf("Отсортированный массив: ");
    for (int i = 0; i < n; ++i) {
        printf("%d ", arr[i]);
    }
    printf("\n");

    // Освобождение динамической памяти
    free(arr);

    return EXIT_SUCCESS;
}
```

---

### Как работает программа?

**1. Подготовка:**  
- Программа запрашивает у пользователя количество элементов `n`. Проверяется корректность ввода.
- С помощью функции `malloc` выделяется блок памяти размером `n * sizeof(int)` — это позволяет работать с переменным размером массива.

**2. Ввод данных:**  
- Пользователь последовательно вводит целые числа, которые записываются по адресу, полученному из динамического выделения.

**3. Пузырьковая сортировка (`bubble_sort`):**
- Внешний цикл проходит от начала до конца массива (всего `n-1` итераций).
- Внутренний цикл сравнивает соседние элементы и меняет их местами, если они стоят не по порядку.
- После каждого прохода самый большой элемент оказывается в конце массива (отсюда название «пузырьковая»).

**4. Вывод результата:**  
- Массив выводится целиком после сортировки.

**5. Очистка ресурсов:**  
- Перед завершением вызывается `free(arr)` для освобождения занятой памяти во избежание утечек.

### Компиляция и запуск
Для сборки достаточно стандартного GCC:
```bash
gcc -o bubble_sort bubble_sort.c
./bubble_sort
```

Если нужно использовать современные стандарты, можно добавить флаг `-std=c11` или выше.

### Особенности реализации:
- Использованы указатели для работы с памятью.
- Есть проверка ошибок ввода и выделения памяти.
- Не используется стандартная библиотека сортировок (`qsort`), реализована именно пузырьковая.

---

## math

## 1. Определение математического ожидания

**Математическое ожидание $E[X]$ случайной величины $X$ — это мера «центральной» или «средней» характеристики распределения вероятностей этой величины.** Оно определяется по-разному для дискретных и непрерывных случайных величин.

---

### 1.1 Дискретная случайная величина

Пусть $X$ — дискретная случайная величина, принимающая значения $x_1, x_2, \dots$ с соответствующими вероятностями $p_i = P(X = x_i)$, где $\sum_{i=1}^{\infty} p_i = 1$.

> **Определение:**  
> $$E[X] = \sum_{i=1}^\infty x_i \, p_i$$

---

### 1.2 Непрерывная случайная величина

Пусть $X$ — непрерывная случайная величина с плотностью вероятности $f(x) \geq 0$, интегрируемой на всей области определения, причём $\int_{-\infty}^{+\infty} f(x)\,dx = 1$.

> **Определение:**  
> $$E[X] = \int_{-\infty}^{+\infty} x\, f(x)\, dx$$

---

## 2. Основные свойства математического ожидания

Свойства $E[\cdot]$ универсальны: они выполняются как для дискретных, так и для непрерывных случайных величин (при условии корректности вычислений).

| № | Свойство | Формула | Краткое доказательство |
|:---:|:---|:---|:---|
| 1 | **Ожидание константы** | $E[c] = c$ (где $c$ — постоянная) | По определению: $E[c] = c \cdot 1 = c$. |
| 2 | **Линейность (аддитивность)** | $E[aX + bY] = a E[X] + b E[Y]$ (для любых констант $a, b$) | Для дискретного случая:  
$$E[aX + bY] = \sum (ax_i + by_i)p_i = a\sum x_ip_i + b\sum y_ip_i = aE[X] + bE[Y].$$ |
| 3 | **Вынос константы за знак ожидания** | $E[aX] = a E[X]$ | Аналогично линейности: $E[aX] = a\sum x_ip_i = aE[X]$. |
| 4 | **Ожидание суммы** | $E[X+Y] = E[X] + E[Y]$ | Аналогично аддитивности. |
| 5 | **Ожидание произведения независимых величин** (если $X$ и $Y$ независимы):  
$$E[XY] = E[X]E[Y]$$ | Для дискретного случая:  
$$E[XY] = \sum x_i y_i p_i = \left(\sum x_i p_i\right)\left(\sum y_j q_j\right) = E[X]E[Y],$$  
где $q_j = P(Y=y_j)$. |
| 6 | **Неотрицательность** (для неотрицательных величин): если $X \geq 0$, то $E[X] \geq 0$ | Прямо из определения: все $x_i \geq 0$ или $x \geq 0$ под интегралом, значит результат не может быть отрицательным. |
| 7 | **Условное математическое ожидание** (свойство маргинальности): $E[E[X|Y]] = E[X]$ | Это следует из формулы полной вероятности. |

---

## 3. Примеры вычисления

### Пример 1: Дискретная случайная величина

**Задача:** Пусть $X$ принимает значения $0, 1, 2$ с вероятностями $P(X=0)=0.2$, $P(X=1)=0.5$, $P(X=2)=0.3$. Найти $E[X]$.

**Решение:**
$$E[X] = 0 \cdot 0.2 + 1 \cdot 0.5 + 2 \cdot 0.3 = 0 + 0.5 + 0.6 = 1.1$$

### Пример 2: Непрерывная случайная величина

**Задача:** Плотность равномерного распределения на отрезке $[0, 1]$ задаётся функцией $f(x) = 1$ при $0 \leq x \leq 1$. Найти $E[X]$.

**Решение:**
$$E[X] = \int_{0}^{1} x \cdot 1\, dx = \left[ \frac{x^2}{2} \right]_0^{1} = \frac{1^2 - 0^2}{2} = \frac{1}{2}$$

---

## Итог

- **Дискретный случай:** $E[X] = \sum x_i p_i$
- **Непрерывный случай:** $E[X] = \int x f(x) dx$
- **Основные свойства:** линейность, ожидание константы, выноса константы, сумма, произведение независимых, неотрицательность.
- **Примеры:** $E[X]=1.1$ для дискретной; $E[X]=0.5$ для равномерного на $[0,1]$.

---

## literature

Рассказ Ивана Сергеевича Тургенева «Муму» — это произведение, в котором ярко отражены социальные проблемы крепостного права и тема человеческой судьбы. Главные герои рассказа олицетворяют разные стороны жизни того времени.

1. Герасим (главный герой)  
   - Кто он: дворник глухонемого крестьянина, крепостной мужик из деревни, отличающийся физической силой, трудолюбием и честностью.  
   - Черты характера: сильный физически, добрый, скромный, терпеливый, преданный своим обязанностям; однако глубоко несчастен и одинок, поскольку лишён возможности свободно выражать свои чувства и мысли.  
   - Роль в сюжете: символизирует беззащитность простого народа перед произволом помещицы и крепостнической системой. Его трагическая судьба раскрывает жестокость и бесчеловечность социального устройства России XIX века.
   - Смысл образа и идея автора: образ Герасима воплощает идею протеста против унижения человеческого достоинства и несправедливости крепостничества. Автор подчёркивает, что даже самый простой человек способен на глубокое чувство и нравственную силу, но бессилен перед жестокой властью и социальными законами. В финале рассказа символический протест героя выражается в его решительном уходе от барыни, показывая стремление к свободе и внутренней независимости.

2. Барыня (помещица)  
   - Кто она: властная, капризная, деспотичная старуха-помещица, живущая в Москве.  
   - Черты характера: высокомерна, своенравна, эгоистична, равнодушна к окружающим людям, склонна к внезапным прихотям и жестоким решениям.  
   - Роль в сюжете: является носителем социальной несправедливости и символом угнетения крепостных крестьян. Именно её воля определяет судьбу героев рассказа, особенно судьбу Муму и самого Герасима.

3. Татьяна (прачка)  
   - Кто она: молодая прачка, робкая и забитая девушка, которой покровительствует Герасим.  
   - Черты характера: покорная, тихая, застенчивая, доверчивая, несмотря на тяжёлую жизнь.  
   - Роль в сюжете: олицетворяет жертвенность и беспомощность простых женщин того времени. Её судьба показывает, как легко можно сломить человека, не обладающего ни силой, ни поддержкой.

4. Капитон Бульба (лакей)  
   - Кто он: лакей, пьяница и неудачливый жених Татьяны.  
   - Черты характера: слабовольный, ленивый, зависимый от алкоголя, неспособный самостоятельно изменить свою жизнь.  
   - Роль в сюжете: служит контрастом Герасиму, подчеркивая его силу и достоинство на фоне общей духовной немощи окружающих.

5. Муму (собачка)  
   - Кто она: маленькая собачка, подаренная Герасиму барыней, ставшая для него единственным близким существом и источником радости.  
   - Черты характера: ласковая, верная, умная, привязанная к хозяину.  
   - Роль в сюжете: выступает символом любви, искренности и чистоты души, противопоставленной холодному миру барской власти. Убийство Муму становится высшей точкой трагедии героя и символом бессмысленности насилия над живым чувством.

Общий смысл образа Герасима заключается в том, чтобы показать трагедию личности, вынужденной жить в условиях бесправия и подавления воли. Через судьбу этого сильного, доброго и чистого душой человека Тургенев доносит мысль о необходимости гуманизма, сострадания и свободы, а также осуждает жестокие устои крепостнического общества.

---

# Работа с DeepSeek через API

Работу с DeepSeek будем проводить через SDK OpenAI (API имеет такой же формат)

## Получение API ключа

[DeepSeek](https://platform.deepseek.com) не предоставляет бесплатных токенов, а так же нет возможности пополнить аккаунт с Российских карт. 

Несколько миллионов токенов были купленны на стронней площадке. С ключем идет `base-url`, к которому следует обращатся, для использования API.

## Получение списка доступных моделей

Так как токены были купленны у агрегатора моделей, в списке будут и другие модели. Отобразим только модели от DeepSeek.

In [20]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://ai.starimg.ru/v1",
)

for m in client.models.list().data:
    if "deepseek" in m.id:
        print(m.id)

deepseek-v4-pro
deepseek-v4.1-flash
deepseek-v4-flash


## Отправка запросов на API

In [21]:
resp = client.chat.completions.create(
    model="deepseek-v4.1-flash",
    messages=[{"role": "user", "content": "Привет, как дела?"}],
)

answer = resp.choices[0].message.content
print(answer)

Привет! У меня всё отлично — готов помочь с чем угодно. А как у тебя дела?


При помощи роли `system` можно скорректировать поведение модели

In [22]:
resp = client.chat.completions.create(
    model="deepseek-v4.1-flash",
    messages=[
        {"role": "system", "content": "Отвечай одним предложением, в конце добавляй смайлик"},
        {"role": "user", "content": "Привет, как дела?"},
    ],
)

answer = resp.choices[0].message.content
print(answer)

Привет, у меня всё отлично, спасибо, а как твои дела? 😊


### Отправим несколько запросов моделям DeepSeek-v4-pro и DeepSeek-v4.1-flash

In [23]:
from IPython.display import Markdown, display

models = ["deepseek-v4-pro", "deepseek-v4.1-flash"]

for model in models:
    display(Markdown(f"# {model}"))
    for prompt in prompts:
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt["prompt"]}],
        )
        answer = resp.choices[0].message.content
        display(Markdown(f"## {prompt["name"]}\n\n{answer}\n\n---"))

# deepseek-v4-pro

## python

```python
import re
from collections import Counter

def count_words(text: str) -> dict:
    """
    Возвращает словарь, в котором ключи — слова, а значения — количество их вхождений.
    Регистр не учитывается, знаки препинания игнорируются.
    """
    # Приводим строку к нижнему регистру
    text = text.lower()
    # Удаляем все символы, которые не являются буквами, цифрами или пробелами
    cleaned_text = re.sub(r'[^\w\s]', '', text)
    # Разбиваем оставшийся текст на слова по пробельным символам
    words = cleaned_text.split()
    # Подсчитываем частоты с помощью Counter и возвращаем как обычный словарь
    return dict(Counter(words))

# Пример вызова
example = "Hello, world! Hello, Python. Python is great, isn't it?"
result = count_words(example)
print(result)
```

**Результат выполнения:**
```
{'hello': 2, 'world': 1, 'python': 2, 'is': 1, 'great': 1, 'isnt': 1, 'it': 1}
```

### Краткое объяснение
1. **Приведение к нижнему регистру** – чтобы слова `"Hello"` и `"hello"` считались одинаковыми.
2. **Удаление знаков препинания** – с помощью регулярного выражения `[^\w\s]` удаляем все символы, не являющиеся буквами, цифрами или пробелами (знаки препинания, включая апострофы и дефисы, исчезают, поэтому `"isn't"` превращается в `"isnt"`).
3. **Разбиение на слова** – метод `split()` делит строку по любым пробельным символам (пробел, табуляция, перенос строки).
4. **Подсчёт частот** – `collections.Counter` автоматически считает количество вхождений каждого элемента списка.

---

## c

```c
#include <stdio.h>
#include <stdlib.h>

// Функция пузырьковой сортировки (оптимизированная версия с флагом)
void bubble_sort(int *arr, int n) {
    int swapped;
    for (int i = 0; i < n - 1; i++) {
        swapped = 0;
        for (int j = 0; j < n - i - 1; j++) {
            if (arr[j] > arr[j + 1]) {
                // Обмен элементов
                int temp = arr[j];
                arr[j] = arr[j + 1];
                arr[j + 1] = temp;
                swapped = 1;
            }
        }
        // Если за проход не было обменов, массив уже отсортирован
        if (!swapped) {
            break;
        }
    }
}

int main(void) {
    int n;

    // 1. Ввод количества элементов
    printf("Введите количество элементов: ");
    if (scanf("%d", &n) != 1 || n <= 0) {
        printf("Ошибка: необходимо ввести положительное целое число.\n");
        return 1;
    }

    // 2. Динамическое выделение памяти
    int *arr = (int*)malloc(n * sizeof(int));
    if (arr == NULL) {
        printf("Ошибка: не удалось выделить память.\n");
        return 1;
    }

    // 3. Ввод элементов массива
    printf("Введите %d целых чисел:\n", n);
    for (int i = 0; i < n; i++) {
        if (scanf("%d", &arr[i]) != 1) {
            printf("Ошибка ввода элемента %d.\n", i + 1);
            free(arr);
            return 1;
        }
    }

    // 4. Сортировка пузырьком
    bubble_sort(arr, n);

    // 5. Вывод результата
    printf("Отсортированный массив:\n");
    for (int i = 0; i < n; i++) {
        printf("%d ", arr[i]);
    }
    printf("\n");

    // 6. Освобождение памяти
    free(arr);

    return 0;
}
```

### Пояснение работы программы

1. **Ввод количества элементов**  
   С помощью `scanf` считывается целое число `n`. Если ввод некорректен или `n <= 0`, программа завершается с сообщением об ошибке.

2. **Динамическое выделение памяти**  
   Функция `malloc` выделяет блок памяти под `n` целых чисел. Если выделение не удалось (возвращён `NULL`), программа сообщает об ошибке и завершается.

3. **Ввод массива**  
   В цикле считываются `n` целых чисел и сохраняются в выделенную память. При ошибке ввода память освобождается, и программа завершается.

4. **Пузырьковая сортировка**  
   Функция `bubble_sort` выполняет сортировку массива по возрастанию. Внешний цикл проходит `n-1` раз, внутренний — попарно сравнивает соседние элементы и меняет их местами, если они стоят в неправильном порядке. Переменная `swapped` используется для раннего выхода: если за полный проход не было ни одного обмена, массив уже отсортирован.

5. **Вывод отсортированного массива**  
   Все элементы выводятся через пробел в одной строке.

6. **Освобождение памяти**  
   Вызов `free(arr)` возвращает выделенную память системе.

Программа компилируется любым компилятором C (например, `gcc -std=c99 -Wall program.c -o program`) и корректно работает с динамической памятью.

---

## math

Пусть \((\Omega,\mathcal F,\mathbb P)\) — вероятностное пространство.  
Общее строгое определение такое:

\[
\mathbb E X=\int_\Omega X(\omega)\,d\mathbb P(\omega),
\]

если интеграл Лебега существует. Для конечного математического ожидания требуется \(\mathbb E|X|<\infty\).

---

## 1. Дискретный случай

Случайная величина \(X\) дискретна, если она принимает не более чем счётное множество значений \(x_1,x_2,\dots\) с вероятностями

\[
p_i=\mathbb P(X=x_i).
\]

Тогда

\[
\boxed{\mathbb E X=\sum_i x_i p_i }
\]

если ряд сходится абсолютно:

\[
\sum_i |x_i|p_i<\infty.
\]

Если абсолютная сходимость отсутствует, математическое ожидание не определено.

---

## 2. Непрерывный случай

Случайная величина \(X\) называется абсолютно непрерывной, если существует плотность \(f_X(x)\ge 0\), такая что для любого борелевского множества \(B\)

\[
\mathbb P(X\in B)=\int_B f_X(x)\,dx.
\]

Тогда математическое ожидание равно

\[
\boxed{\mathbb E X=\int_{-\infty}^{\infty} x f_X(x)\,dx }
\]

если

\[
\int_{-\infty}^{\infty} |x| f_X(x)\,dx<\infty.
\]

---

# Основные свойства

Будем предполагать, что указанные математические ожидания существуют.

### 1. Математическое ожидание константы

Если \(X=c\) почти наверное, то

\[
\mathbb E X=c.
\]

**Доказательство.**  
Величина \(X\) дискретна, принимает единственное значение \(c\) с вероятностью \(1\), поэтому

\[
\mathbb E X=c\cdot \mathbb P(X=c)=c\cdot 1=c.
\]

---

### 2. Линейность

Для любых случайных величин \(X,Y\) и чисел \(a,b\)

\[
\mathbb E(aX+bY)=a\mathbb E X+b\mathbb E Y.
\]

**Доказательство в дискретном случае.**  
Пусть \(X\) принимает значения \(x_i\), \(Y\) — значения \(y_j\). Обозначим

\[
p_{ij}=\mathbb P(X=x_i,Y=y_j).
\]

Тогда

\[
\begin{aligned}
\mathbb E(aX+bY)
&=\sum_{i,j}(a x_i+b y_j)p_{ij}\\
&=a\sum_i x_i\sum_j p_{ij}
+b\sum_j y_j\sum_i p_{ij}\\
&=a\sum_i x_i\mathbb P(X=x_i)
+b\sum_j y_j\mathbb P(Y=y_j)\\
&=a\mathbb E X+b\mathbb E Y.
\end{aligned}
\]

**Доказательство в непрерывном случае.**  
Если \((X,Y)\) имеет совместную плотность \(f_{X,Y}(x,y)\), то

\[
\begin{aligned}
\mathbb E(aX+bY)
&=\iint (a x+b y)f_{X,Y}(x,y)\,dx\,dy\\
&=a\int x\left(\int f_{X,Y}(x,y)\,dy\right)dx
+b\int y\left(\int f_{X,Y}(x,y)\,dx\right)dy\\
&=a\int x f_X(x)\,dx+b\int y f_Y(y)\,dy\\
&=a\mathbb E X+b\mathbb E Y.
\end{aligned}
\]

---

### 3. Неотрицательность и монотонность

Если \(X\ge 0\) почти наверное, то

\[
\mathbb E X\ge 0.
\]

Если \(X\le Y\) почти наверное, то

\[
\mathbb E X\le \mathbb E Y.
\]

**Доказательство.**  
Для неотрицательной дискретной величины все \(x_i\ge 0\) с положительной вероятностью, поэтому сумма неотрицательна. Для непрерывной величины плотность \(f_X(x)=0\) при \(x<0\), поэтому интеграл от \(x f_X(x)\) неотрицателен.

Если \(X\le Y\), то \(Y-X\ge 0\). Тогда

\[
\mathbb E(Y-X)\ge 0.
\]

По линейности

\[
\mathbb E Y-\mathbb E X\ge 0,
\]

то есть

\[
\mathbb E X\le \mathbb E Y.
\]

---

### 4. Математическое ожидание функции от случайной величины

Если \(g:\mathbb R\to\mathbb R\) — борелевская функция, то

\[
\mathbb E g(X)=
\begin{cases}
\sum_i g(x_i)\mathbb P(X=x_i), & X\text{ дискретна},\\[4pt]
\displaystyle\int_{-\infty}^{\infty} g(x)f_X(x)\,dx, & X\text{ непрерывна},
\end{cases}
\]

при условии абсолютной сходимости суммы или интеграла.

Это свойство часто называют «правилом бессознательного статистика».

---

### 5. Математическое ожидание произведения независимых величин

Если \(X\) и \(Y\) независимы и \(\mathbb E|X|<\infty\), \(\mathbb E|Y|<\infty\), то

\[
\mathbb E(XY)=\mathbb E X\,\mathbb E Y.
\]

**Доказательство в дискретном случае.**  
Пусть \(X\) принимает значения \(x_i\), а \(Y\) — значения \(y_j\). По независимости

\[
\mathbb P(X=x_i,Y=y_j)=\mathbb P(X=x_i)\mathbb P(Y=y_j).
\]

Тогда

\[
\begin{aligned}
\mathbb E(XY)
&=\sum_{i,j}x_i y_j\mathbb P(X=x_i,Y=y_j)\\
&=\sum_{i,j}x_i y_j\mathbb P(X=x_i)\mathbb P(Y=y_j)\\
&=\left(\sum_i x_i\mathbb P(X=x_i)\right)
\left(\sum_j y_j\mathbb P(Y=y_j)\right)\\
&=\mathbb E X\,\mathbb E Y.
\end{aligned}
\]

**Доказательство в непрерывном случае.**  
Если \(X\) и \(Y\) независимы и имеют плотности \(f_X,f_Y\), то совместная плотность равна

\[
f_{X,Y}(x,y)=f_X(x)f_Y(y).
\]

Следовательно,

\[
\begin{aligned}
\mathbb E(XY)
&=\iint xy f_X(x)f_Y(y)\,dx\,dy\\
&=\left(\int x f_X(x)\,dx\right)
\left(\int y f_Y(y)\,dy\right)\\
&=\mathbb E X\,\mathbb E Y.
\end{aligned}
\]

---

### 6. Формула для дисперсии

Если \(\mathbb E X^2<\infty\), то

\[
\operatorname{Var}X=\mathbb E[(X-\mathbb E X)^2]
=\mathbb E X^2-(\mathbb E X)^2.
\]

**Доказательство.**  
Обозначим \(\mu=\mathbb E X\). Тогда

\[
\begin{aligned}
\mathbb E[(X-\mu)^2]
&=\mathbb E[X^2-2\mu X+\mu^2]\\
&=\mathbb E X^2-2\mu\mathbb E X+\mu^2\\
&=\mathbb E X^2-2\mu^2+\mu^2\\
&=\mathbb E X^2-\mu^2.
\end{aligned}
\]

---

# Примеры

## Дискретный пример

Пусть \(X\) — число очков при броске правильной шестигранной кости. Тогда

\[
\mathbb P(X=k)=\frac16,\qquad k=1,2,\dots,6.
\]

Следовательно,

\[
\mathbb E X
=\sum_{k=1}^6 k\cdot \frac16
=\frac{1+2+3+4+5+6}{6}
=\frac{21}{6}
=\frac72
=3{,}5.
\]

---

## Непрерывный пример

Пусть \(X\sim \mathrm{Exp}(\lambda)\), \(\lambda>0\). Плотность:

\[
f_X(x)=
\begin{cases}
\lambda e^{-\lambda x}, & x\ge 0,\\
0, & x<0.
\end{cases}
\]

Тогда

\[
\mathbb E X
=\int_0^\infty x\lambda e^{-\lambda x}\,dx.
\]

Сделаем замену \(u=\lambda x\), тогда \(dx=\frac{du}{\lambda}\). Получаем

\[
\mathbb E X
=\frac1\lambda\int_0^\infty u e^{-u}\,du.
\]

Интеграл

\[
\int_0^\infty u e^{-u}\,du=1,
\]

поэтому

\[
\mathbb E X=\frac1\lambda.
\]

Например, при \(\lambda=1\) имеем \(\mathbb E X=1\).

---

## literature

## Главные действующие лица рассказа И. С. Тургенева «Муму»

### Герасим

**Кто он:** глухонемой дворник, крепостной крестьянин, привезённый из деревни в московский дом старой барыни. Мужчина огромного роста и необыкновенной физической силы.

**Черты характера:** трудолюбивый, добрый, честный, верный, способный на глубокую и преданную привязанность. Несмотря на физический недостаток, он наделён душевной чуткостью и чувством собственного достоинства. В нём сочетаются богатырская мощь и нравственная чистота, но вместе с тем — привычка к беспрекословному повиновению.

**Роль в сюжете:** центральный герой. Его жизнь, любовь к прачке Татьяне, привязанность к собачке Муму и трагическое подчинение барскому приказу раскрывают главный конфликт рассказа — столкновение живого человеческого чувства с жестокой крепостнической системой.

### Барыня

**Кто она:** старая помещица-вдова, владелица многочисленных крепостных. Живёт в Москве, окружённая прислугой и приживалками.

**Черты характера:** капризная, своенравная, деспотичная, равнодушная к судьбам своих крепостных. Её жизнь пуста и скучна, поэтому она развлекается, распоряжаясь людьми, не задумываясь о последствиях своих прихотей. Она не столько зла, сколько привыкла к полной власти над людьми.

**Роль в сюжете:** олицетворяет помещичий произвол. Её капризы становятся движущей силой трагических событий: сначала она выдаёт Татьяну замуж за пьяницу Капитона, затем требует избавиться от Муму. Именно её воля разрушает счастье Герасима.

### Татьяна

**Кто она:** крепостная прачка, женщина лет двадцати восьми.

**Черты характера:** тихая, робкая, запуганная, безропотная. Всю жизнь она привыкла подчиняться и не имеет сил сопротивляться. Она трудолюбива, но её душевная подавленность не позволяет ей даже ответить на любовь Герасима.

**Роль в сюжете:** объект первой привязанности Герасима. Её насильственная выдача замуж за Капитона показывает полное бесправие крепостных людей и становится для Герасима первым тяжёлым ударом.

### Капитон Климов

**Кто он:** башмачник, крепостной, горький пьяница.

**Черты характера:** никчёмный, ленивый, слабохарактерный, но при этом с претензией на «образованность» и «тонкость чувств». Он не злой, но жалкий человек, сломленный крепостной действительностью.

**Роль в сюжете:** его женитьба на Татьяне, устроенная по приказу барыни, окончательно лишает Герасима надежды на личное счастье. Капитон — ещё одна жертва крепостного права, хотя видимой причиной бед становится именно он.

### Гаврила

**Кто он:** дворецкий в доме барыни.

**Черты характера:** исполнительный, трусливый, угодливый. Он не имеет собственной воли и лишь следит за тем, чтобы приказы барыни выполнялись точно и без возражений.

**Роль в сюжете:** посредник между барыней и остальными крепостными. Именно через него передаётся приказ об уничтожении Муму.

### Муму

**Кто она:** маленькая собачка, спасённая Герасимом из реки и выкормленная им.

**Черты характера:** верная, ласковая, беззащитная, преданная хозяину.

**Роль в сюжете:** становится для Герасима тем единственным существом, которое бескорыстно любит его. Её гибель — кульминация рассказа, момент, когда герой теряет всё, что ему дорого. Муму — символ живого чувства, безжалостно уничтоженного барской волей.

---

## Смысл образа Герасима и идея автора

Образ Герасима — один из самых значимых в русской литературе. Это **олицетворение русского народа**: внешне могучий, трудолюбивый, нравственно здоровый, но в условиях крепостного права — бесправный и немой. Его глухонемота — не только физический недостаток, но и **символ подневольного молчания**, невозможности открыто выразить протест. Он не может говорить, но чувствует глубоко и сильно.

Герасим наделён чертами былинного богатыря, однако его сила не находит достойного применения: он вынужден подчиняться прихотям капризной старухи. Любовь к Татьяне и привязанность к Муму показывают, что он способен на искренние, глубокие чувства. Однако система, в которой он живёт, отнимает у него сначала любимую женщину, а затем и единственное живое существо, отвечавшее ему любовью.

Убийство Муму — трагический перелом. Герасим впервые проявляет самостоятельность: он подчиняется приказу, но затем **уходит от барыни обратно в деревню**. Этот шаг — стихийный, молчаливый, но мощный протест против произвола. Он не может открыто восстать, но он больше не желает оставаться рабом.

Тургенев показывает, что даже самый покорный человек имеет предел терпения. Писатель осуждает крепостное право, которое калечит судьбы, лишает людей свободы, любви и человеческого достоинства. Вместе с тем в образе Герасима автор выражает веру в скрытую силу народа, которая рано или поздно должна проснуться. Идея рассказа — протест против угнетения человека человеком и утверждение права каждой личности на чувство, свободу и внутреннюю независимость.

---

# deepseek-v4.1-flash

## python

```python
import re
from collections import Counter

def word_count(text: str) -> dict[str, int]:
    words = re.findall(r'[^\W_]+', text.lower(), flags=re.UNICODE)
    return dict(Counter(words))
```

**Кратко как работает:**
- `text.lower()` — приводит строку к нижнему регистру.
- `re.findall(r'[^\W_]+', ...)` — находит последовательности букв и цифр, игнорируя знаки препинания и подчёркивания.
- `Counter(words)` — подсчитывает количество каждого слова.
- `dict(...)` — возвращает обычный словарь.

**Пример вызова:**

```python
text = "Привет, мир! Привет, Python. Python — это мир."
print(word_count(text))
```

**Результат:**

```python
{'привет': 2, 'мир': 2, 'python': 2, 'это': 1}
```

---

## c

```c
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    int n;
    printf("Введите количество элементов: ");
    if (scanf("%d", &n) != 1 || n <= 0) {
        fprintf(stderr, "Некорректное значение n\n");
        return 1;
    }

    int *a = malloc((size_t)n * sizeof(int));
    if (a == NULL) {
        fprintf(stderr, "Ошибка выделения памяти\n");
        return 1;
    }

    printf("Введите %d целых чисел:\n", n);
    for (int i = 0; i < n; i++) {
        if (scanf("%d", &a[i]) != 1) {
            fprintf(stderr, "Ошибка ввода\n");
            free(a);
            return 1;
        }
    }

    /* Пузырьковая сортировка */
    for (int i = 0; i < n - 1; i++) {
        int swapped = 0;
        for (int j = 0; j < n - 1 - i; j++) {
            if (a[j] > a[j + 1]) {
                int tmp = a[j];
                a[j] = a[j + 1];
                a[j + 1] = tmp;
                swapped = 1;
            }
        }
        if (!swapped) break; /* массив уже отсортирован */
    }

    printf("Отсортированный массив:\n");
    for (int i = 0; i < n; i++) {
        printf("%d ", a[i]);
    }
    printf("\n");

    free(a);
    return 0;
}
```

### Как работает программа

1. **Ввод `n`.** Считываем количество элементов и проверяем, что оно положительное.
2. **Выделение памяти.** Через `malloc` выделяем `n * sizeof(int)` байт под массив. Проверяем, что память выделилась (`a != NULL`).
3. **Чтение массива.** В цикле считываем `n` целых чисел в массив. При ошибке ввода освобождаем память и завершаем работу.
4. **Пузырьковая сортировка.**
   - Внешний цикл `i` — номер прохода. После каждого прохода наибольший из ещё не отсортированных элементов «всплывает» в конец.
   - Внутренний цикл `j` идёт до `n - 1 - i`, потому что последние `i` элементов уже стоят на своих местах.
   - Если соседние элементы стоят в неправильном порядке (`a[j] > a[j+1]`), меняем их местами и ставим флаг `swapped = 1`.
   - Если за проход не было ни одной перестановки — массив уже отсортирован, выходим из внешнего цикла (оптимизация).
5. **Вывод результата.** Печатаем отсортированный массив.
6. **Освобождение памяти.** Вызываем `free(a)`, чтобы не было утечки.

Сложность алгоритма: O(n²) в худшем случае, O(n) — в лучшем (уже отсортированный массив).

---

## math

## 1. Строгое определение

Пусть задано вероятностное пространство \((\Omega,\mathcal F,\mathbb P)\), а \(X:\Omega\to\mathbb R\) — случайная величина. В общем случае математическое ожидание определяется как интеграл Лебега
\[
\mathbb E X=\int_\Omega X(\omega)\,d\mathbb P(\omega),
\]
если \(\mathbb E|X|=\int_\Omega |X|\,d\mathbb P<\infty\). Если интеграл не абсолютно сходится, математическое ожидание в стандартном смысле не определено (для неотрицательных величин допускается значение \(+\infty\)).

### Дискретный случай

Пусть \(X\) принимает не более чем счётное множество значений \(x_1,x_2,\dots\) с вероятностями
\[
p_i=\mathbb P(X=x_i),\qquad \sum_i p_i=1.
\]
Тогда
\[
\boxed{\mathbb E X=\sum_i x_i p_i}
\]
при условии
\[
\sum_i |x_i|p_i<\infty.
\]
Если \(X\ge 0\), сумма может быть и бесконечной.

### Непрерывный случай

Пусть \(X\) — абсолютно непрерывная случайная величина с плотностью \(f_X(x)\). Тогда
\[
\boxed{\mathbb E X=\int_{-\infty}^{\infty} x f_X(x)\,dx}
\]
при условии
\[
\int_{-\infty}^{\infty} |x|f_X(x)\,dx<\infty.
\]
Если \(X\ge 0\), интеграл может быть равен \(+\infty\).

---

## 2. Основные свойства математического ожидания

### 1) Ожидание константы
Если \(X\equiv c\) почти наверное, то
\[
\mathbb E[c]=c.
\]
Доказательство:
\[
\mathbb E[c]=\int_\Omega c\,d\mathbb P=c\mathbb P(\Omega)=c.
\]

### 2) Линейность
Если \(\mathbb E|X|<\infty\), \(\mathbb E|Y|<\infty\), то для любых \(a,b\in\mathbb R\)
\[
\mathbb E[aX+bY]=a\mathbb E X+b\mathbb E Y.
\]
Доказательство: по линейности интеграла Лебега
\[
\mathbb E[aX+bY]=\int_\Omega (aX+bY)\,d\mathbb P
=a\int_\Omega X\,d\mathbb P+b\int_\Omega Y\,d\mathbb P
=a\mathbb E X+b\mathbb E Y.
\]
В дискретном случае это сумма по совместному распределению, в непрерывном — двойной интеграл по совместной плотности. Независимость здесь не нужна.

### 3) Монотонность
Если \(X\le Y\) почти наверное, то
\[
\mathbb E X\le \mathbb E Y.
\]
Доказательство: \(Y-X\ge 0\) почти наверное, а математическое ожидание неотрицательной величины неотрицательно:
\[
\mathbb E Y-\mathbb E X=\mathbb E[Y-X]\ge 0.
\]

### 4) Ожидание индикатора
Для события \(A\in\mathcal F\)
\[
\mathbb E[\mathbf 1_A]=\mathbb P(A).
\]
Доказательство:
\[
\mathbb E[\mathbf 1_A]=\int_\Omega \mathbf 1_A\,d\mathbb P=\mathbb P(A).
\]

### 5) Ожидание произведения независимых величин
Если \(X\) и \(Y\) независимы и \(\mathbb E|X|<\infty\), \(\mathbb E|Y|<\infty\), то
\[
\mathbb E[XY]=\mathbb E X\cdot \mathbb E Y.
\]
Доказательство в дискретном случае. Пусть
\[
p_i=\mathbb P(X=x_i),\qquad q_j=\mathbb P(Y=y_j).
\]
Тогда в силу независимости
\[
\mathbb P(X=x_i,Y=y_j)=p_iq_j.
\]
Следовательно,
\[
\mathbb E[XY]=\sum_{i,j}x_iy_jp_iq_j
=\left(\sum_i x_ip_i\right)\left(\sum_j y_jq_j\right)
=\mathbb E X\cdot \mathbb E Y.
\]
В непрерывном случае, если \(f_{X,Y}(x,y)=f_X(x)f_Y(y)\),
\[
\mathbb E[XY]=\int\!\!\int xy\,f_X(x)f_Y(y)\,dx\,dy
=\left(\int x f_X(x)\,dx\right)\left(\int y f_Y(y)\,dy\right).
\]

### 6) Ожидание функции от случайной величины
В дискретном случае
\[
\mathbb E[g(X)]=\sum_i g(x_i)p_i,
\]
в непрерывном случае
\[
\mathbb E[g(X)]=\int_{-\infty}^{\infty} g(x)f_X(x)\,dx,
\]
если соответствующий ряд или интеграл сходится абсолютно.

---

## 3. Примеры вычисления

### Дискретный пример: честный кубик

Пусть \(X\) — число выпавших очков на честном игральном кубике. Тогда
\[
\mathbb P(X=k)=\frac16,\qquad k=1,2,\dots,6.
\]
Следовательно,
\[
\mathbb E X=\sum_{k=1}^6 k\cdot \frac16
=\frac{1+2+3+4+5+6}{6}
=\frac{21}{6}=3{,}5.
\]

### Непрерывный пример: показательное распределение

Пусть \(X\sim \mathrm{Exp}(\lambda)\), \(\lambda>0\). Её плотность:
\[
f_X(x)=
\begin{cases}
\lambda e^{-\lambda x}, & x\ge 0,\\
0, & x<0.
\end{cases}
\]
Тогда
\[
\mathbb E X=\int_0^\infty x\lambda e^{-\lambda x}\,dx.
\]
Интегрируя по частям или заменой \(u=\lambda x\),
\[
\mathbb E X=\frac1\lambda.
\]
Например, при \(\lambda=2\) имеем \(\mathbb E X=0{,}5\).

---

## literature

«Муму» — рассказ о крепостной неволе, в центре которого трагическая судьба глухонемого дворника Герасима. Главные действующие лица — Герасим, Муму, барыня, Татьяна, Капитон и дворецкий Гаврила.

**Герасим**  
Кто он: крепостной крестьянин, глухонемой от рождения, огромной физической силы, привезённый барыней из деревни в Москву и назначенный дворником.  
Черты характера: трудолюбивый, честный, суровый внешне, но добрый и чуткий; замкнутый, одинокий, терпеливый, способный на глубокую любовь и привязанность. При этом в нём есть чувство собственного достоинства.  
Роль в сюжете: центральный герой. Его судьба организует всё повествование. Любовь к Татьяне и привязанность к Муму — единственные светлые чувства в его жизни. По прихоти барыни он теряет Татьяну, а потом вынужден убить Муму. После этого уходит из Москвы в деревню. Это кульминация и развязка рассказа.

**Муму**  
Кто она: маленькая собачка, спасённая Герасимом из реки.  
Черты: преданная, ласковая, беззащитная, привязанная только к Герасиму.  
Роль в сюжете: символ единственного тепла и любви в жизни Герасима. Её появление даёт ему счастье, а конфликт с барыней и гибель становятся страшным обвинением крепостничеству. Муму — невинная жертва барского самодурства.

**Барыня**  
Кто она: старая московская помещица, владелица Герасима и других крепостных.  
Черты: деспотичная, капризная, эгоистичная, мнительная, жестокая. Её чувства — это прихоти, она не считается с живыми людьми.  
Роль в сюжете: главная антагонистка. Именно её произвол ломает судьбы Татьяны, Капитона и Герасима, губит Муму. Она олицетворяет бесконтрольную власть крепостного права.

**Татьяна**  
Кто она: молодая прачка, крепостная барыни.  
Черты: тихая, кроткая, трудолюбивая, запуганная, покорная. Она привыкла терпеть и не смеет сопротивляться.  
Роль в сюжете: объект любви Герасима. По приказу барыни её выдают замуж за Капитона, и это становится для Герасима тяжёлым ударом. Её судьба показывает полное бесправие крепостных в личной жизни.

**Капитон Климов**  
Кто он: сапожник, крепостной, горький пьяница.  
Черты: слабохарактерный, распущенный, трусливый, жалкий; пытается казаться образованным.  
Роль в сюжете: муж Татьяны. Его женитьба на ней разлучает её с Герасимом. Пьянство и деградация Капитона — следствие жизни в крепостной неволе, хотя и сам он изображён неприятно.

**Гаврила Андреевич**  
Кто он: дворецкий, главный управляющий дворни барыни.  
Черты: хитрый, услужливый, боязливый перед барыней, готовый выполнить любой её приказ. Сам крепостной, но угнетает других.  
Роль в сюжете: посредник между барыней и дворней. Именно он передаёт Герасиму приказ избавиться от Муму. Через этот образ показано, как крепостническая система развращает и самих слуг.

**Смысл образа Герасима и идея автора**

Герасим — воплощение лучших черт русского народа: богатырской силы, трудолюбия, честности, доброты и способности к глубокой любви. Но он глухонемой и крепостной. Его немота символична: он не может сказать «нет», не может открыто протестовать, его сила и чувства принадлежат барыне. Он терпит, пока хватает сил: теряет Татьяну, потом Муму. Убийство Муму он совершает сам, потому что не смеет ослушаться, но после этого его терпение кончается. Уход в деревню — это немой бунт, нравственная победа и отказ быть рабом.

Тургенев хотел показать, что крепостное право калечит и губит людей, разрывает естественные человеческие связи, превращает одних в жертв, других — в палачей и прислужников. Автор осуждает деспотизм и безнаказанную власть помещиков, но вместе с тем сочувствует простому народу и верит в его нравственную силу. Образ Герасима — это не только обвинение крепостничеству, но и утверждение человеческого достоинства, которое нельзя уничтожить до конца.

---

# Краткие итоги

GigaChat и DeepSeek оба предоставляют удобное и очень схожее API.  

У DeepSeek покупка токенов в РФ менее удобна.